# Yelp Review Fraud Detection — Training Notebook

BT4012 project. This notebook trains the GPU and neural parts of the design and hands **raw outputs** to `pipeline.ipynb`, which calibrates, thresholds and evaluates them like every other detector. It never re-splits the data, never calibrates and never looks at test labels.

| Section | Runtime | Output on Drive | Used by |
| --- | --- | --- | --- |
| T0 Setup | CPU | (checks only) | everything below |
| T1 Embeddings | T4 | `artifacts/embeddings/<mode>/<model>/` | T2 only |
| T2 Similarity | T4 (CPU for debugging) | `data/similarity_features.parquet`, `artifacts/neighbours.parquet` | D2 features, D4 (optional), `find_similar_reviews` |
| T3 DistilBERT | T4 | `artifacts/raw_scores/d3_distilbert.parquet` | D3 *(next)* |
| T4 1D CNN | CPU | `artifacts/raw_scores/d2_cnn.parquet` | D2 *(after the pipeline builds D2's features)* |

**Inputs** (written by `pipeline.ipynb` Section 1): `data/reviews_clean.parquet`, `data/splits.parquet`, `config.json`.

**How to run**

1. **Debug first, on CPU.** Leave `RUN_FULL = False` in T0.2. T1–T2 then run on about 2,000 dev-sample reviews with a small embedding model, writing to separate `sample` files.
2. **Real run, on a T4.** *Runtime → Change runtime type → T4 GPU*, set `RUN_FULL = True`, run T0 → T2. T1 saves embeddings in 50k-review chunks, so a dropped session resumes where it stopped.
3. **Disconnect and delete the runtime** as soon as the run ends; units burn while it sits idle.

## T0. Setup
### T0.1 Install dependencies
Colab ships PyTorch, pandas and NumPy; `sentence-transformers` loads the embedding models from Hugging Face.

In [2]:
%pip install -q -U sentence-transformers

### T0.2 Config, Drive and runtime
Shared settings (seed, paths) come from the pipeline's `config.json`, so both notebooks agree. Settings that only matter here live in `TrainConfig` and are saved to `artifacts/train_config.json` with each run.

`RUN_FULL` decides the mode:

- `False` (debug): dev-sample reviews; on CPU, capped at `cpu_debug_rows` and encoded with the small `debug_model`.
- `True` (real run): all reviews; requires a GPU.

In [1]:
import os, sys, json, random, time, platform, hashlib
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

RUN_FULL = True

PROJECT_ROOT = Path(os.environ.get("BT4012_ROOT", "/content/drive/MyDrive/bt4012_yelp"))
if str(PROJECT_ROOT).startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

shared = json.loads((PROJECT_ROOT / "config.json").read_text())      # written by pipeline.ipynb 0.2
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class TrainConfig:
    seed: int = shared.get("seed", 42)
    # T1 embeddings
    embed_candidates: tuple = ("BAAI/bge-m3", "BAAI/bge-base-en-v1.5")   # benchmarked in T1.1, in order of preference
    embed_budget_min: float = 90.0          # pick the first candidate projected to embed everything within this
    debug_model: str = "BAAI/bge-small-en-v1.5"   # CPU smoke tests only
    max_seq_len: int = 256                  # tokens; longer reviews are truncated
    batch_size: int = 128
    chunk_size: int = 50_000                # reviews per saved chunk (resume unit)
    cpu_debug_rows: int = 2_000
    # T2 similarity
    knn_k: int = 20                         # neighbours kept per review (other accounts only)
    knn_block: int = 1024                   # query rows per GPU matmul
    n_neighbours_saved: int = 10            # neighbours kept for find_similar_reviews
    tau_grid: tuple = (0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.99)
    min_lift: float = 1.5                   # tau rule: broadest cutoff with training lift >= this (as D1)
    tau_copy: float = 0.97                  # fixed cutoff for "near copy": matches below it read as generic wording


TC = TrainConfig()
MODE = "full" if RUN_FULL else "sample"
SUFFIX = "" if RUN_FULL else "_sample"      # sample outputs never overwrite full ones

os.environ["PYTHONHASHSEED"] = str(TC.seed)
random.seed(TC.seed)
np.random.seed(TC.seed)
torch.manual_seed(TC.seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gpu = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
print(f"Mode: {MODE} | device: {DEVICE}{f' ({gpu})' if gpu else ''} | torch {torch.__version__}")
if RUN_FULL and DEVICE != "cuda":
    print("No GPU: a full run can only reuse saved embeddings and neighbours (T1.2 and T2.1 stop if they are missing).")
if gpu and "T4" not in gpu:
    print(f"WARNING: {gpu} burns Colab units faster than a T4; switch unless a T4 is unavailable.")

(ARTIFACTS_DIR / "train_config.json").write_text(json.dumps({**asdict(TC), "mode": MODE}, indent=2))

Mounted at /content/drive
Mode: full | device: cuda (Tesla T4) | torch 2.11.0+cu130


464

### T0.3 Load data and check the contract
Reads the pipeline's cleaned reviews and splits, and stops if they disagree: training here must use exactly the reviews and splits the pipeline evaluates on.

`work` is the table T1–T2 operate on: all reviews in a full run, the dev sample (whole reviewers) when debugging. Its row order, sorted by `review_id`, is the row order of the embedding matrix.

In [3]:
COLS = ["review_id", "reviewer_id", "restaurant_id", "text", "is_fraud", "word_count", "text_key",
        "copy_other_accounts"]
reviews = pd.read_parquet(DATA_DIR / "reviews_clean.parquet", columns=COLS)
splits = pd.read_parquet(DATA_DIR / "splits.parquet")

if set(reviews["review_id"]) != set(splits["review_id"]):
    raise RuntimeError("reviews_clean.parquet and splits.parquet disagree; re-run pipeline.ipynb Section 1.")
d = reviews.merge(splits, on=["review_id", "reviewer_id"], validate="one_to_one")
assert d.groupby("reviewer_id")["split"].nunique().max() == 1, "a reviewer appears in more than one split"

print(f"{len(d):,} reviews, {d['reviewer_id'].nunique():,} reviewers, "
      f"{d['restaurant_id'].nunique():,} restaurants")
display(d.groupby("split").agg(reviews=("review_id", "size"), fraud_rate=("is_fraud", "mean"))
        .reindex(["train", "val", "test"]).style.format({"reviews": "{:,}", "fraud_rate": "{:.2%}"}))

if RUN_FULL:
    work = d
else:
    work = d[d["in_dev_sample"]]
    if DEVICE == "cpu":                       # whole reviewers, up to cpu_debug_rows reviews
        rv = work["reviewer_id"].drop_duplicates().sample(frac=1, random_state=TC.seed)
        sizes = work["reviewer_id"].value_counts().reindex(rv).cumsum()
        work = work[work["reviewer_id"].isin(sizes[sizes <= TC.cpu_debug_rows].index)]
work = work.sort_values("review_id").reset_index(drop=True)
print(f"\nWorking set ({MODE}): {len(work):,} reviews from {work['reviewer_id'].nunique():,} reviewers")

608,598 reviews, 260,277 reviewers, 5,044 restaurants


,reviews,fraud_rate
split,,
train,"426,331",13.24%
val,"90,530",13.34%
test,"91,737",13.02%



Working set (full): 608,598 reviews from 260,277 reviewers


### T0.4 Shared helpers
- `save_table`: writes a parquet file plus a `.meta.json` beside it (what made it, when, with which settings).
- `save_raw_scores`: the hand-off format for T3 and T4. One row per review with `review_id, score, evidence`, checked against the reviews before saving. The pipeline reads it through a small wrapper detector and calibrates it there.

In [4]:
from datetime import datetime, timezone

RAW_SCORE_DIR = ARTIFACTS_DIR / "raw_scores"


def save_table(df: pd.DataFrame, path: Path, meta: dict) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    meta = {**meta, "rows": len(df), "columns": list(df.columns), "mode": MODE,
            "created_at": datetime.now(timezone.utc).isoformat()}
    path.with_suffix(".meta.json").write_text(json.dumps(meta, indent=2, default=str))
    return path


def save_raw_scores(name: str, out: pd.DataFrame, meta: dict) -> Path:
    '''Raw detector scores for pipeline.ipynb: review_id, score (higher = more suspicious), evidence.'''
    if "evidence" not in out:
        out = out.assign(evidence=None)
    out = out[["review_id", "score", "evidence"]]
    expected = set(work["review_id"])
    if len(out) != len(expected) or set(out["review_id"]) != expected:
        raise ValueError(f"{name}: need exactly one row per review in the working set")
    if not np.isfinite(out["score"].to_numpy(dtype=float)).all():
        raise ValueError(f"{name}: scores contain NaN or inf")
    return save_table(out, RAW_SCORE_DIR / f"{name}{SUFFIX}.parquet", {"detector": name, **meta})

## T1. Review embeddings
**Objective:** one normalised vector per review, so that reviews with the same meaning (not just the same words) end up close together. T2 turns these into similarity features; nothing else reads them directly.

### T1.1 Choose the model by benchmark (GPU only)
Times each candidate on 2,000 reviews and projects the time for every review. The first candidate within `embed_budget_min` minutes is used; set `EMBED_MODEL` to override. On CPU this is skipped and the small debug model is used.

In [6]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"     # <- None = use the benchmark's pick; or e.g. "BAAI/bge-base-en-v1.5"


def load_encoder(name: str) -> SentenceTransformer:
    m = SentenceTransformer(name, device=DEVICE)
    m.max_seq_length = TC.max_seq_len
    if DEVICE == "cuda":
        m.half()
    return m


def encode(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    return model.encode(texts, batch_size=TC.batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False).astype(np.float16)


if DEVICE == "cuda":
    bench_texts = d["text"].sample(2_000, random_state=TC.seed).tolist()
    rows = []
    for name in TC.embed_candidates:
        m = load_encoder(name)
        encode(m, bench_texts[:256])                       # warm-up
        torch.cuda.synchronize(); t0 = time.time()
        e = encode(m, bench_texts)
        torch.cuda.synchronize(); rate = len(bench_texts) / (time.time() - t0)
        rows.append({"model": name, "dim": e.shape[1], "reviews/s": rate, "projected min (all reviews)": len(d) / rate / 60})
        del m; torch.cuda.empty_cache()
    bench = pd.DataFrame(rows).set_index("model")
    display(bench.style.format({"reviews/s": "{:,.0f}", "projected min (all reviews)": "{:.0f}"}))
    fits = bench.index[bench["projected min (all reviews)"] <= TC.embed_budget_min]
    pick = fits[0] if len(fits) else bench["projected min (all reviews)"].idxmin()
else:
    pick = TC.debug_model
    print("No GPU: skipping the benchmark" + ("." if EMBED_MODEL or RUN_FULL else "; the small debug model is used."))
MODEL_NAME = EMBED_MODEL or pick
print(f"Embedding model: {MODEL_NAME}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (22 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,dim,reviews/s,projected min (all reviews)
model,,,
BAAI/bge-m3,1024,176,58
BAAI/bge-base-en-v1.5,768,473,21


Embedding model: BAAI/bge-m3


### T1.2 Encode every review (resumable)
Reviews are encoded in chunks of `chunk_size`, each saved to Drive as soon as it finishes; a re-run skips chunks already on disk, so a dropped session costs at most one chunk. Each model and mode gets its own folder, and the cell stops if the saved review IDs no longer match the working set.

In [7]:
slug = MODEL_NAME.replace("/", "__")
EMB_DIR = ARTIFACTS_DIR / "embeddings" / MODE / slug
EMB_DIR.mkdir(parents=True, exist_ok=True)
ids_path = EMB_DIR / "embedding_ids.parquet"

ids_hash = hashlib.sha1(work["review_id"].astype(str).str.cat(sep="|").encode()).hexdigest()
if ids_path.exists():
    saved = json.loads((EMB_DIR / "embedding_meta.json").read_text()) if (EMB_DIR / "embedding_meta.json").exists() else {}
    if saved.get("ids_hash", ids_hash) != ids_hash:
        raise RuntimeError(f"Saved embeddings in {EMB_DIR} are for a different set of reviews; delete the folder to re-embed.")
work[["review_id"]].to_parquet(ids_path, index=False)
(EMB_DIR / "embedding_meta.json").write_text(json.dumps(
    {"model": MODEL_NAME, "max_seq_len": TC.max_seq_len, "normalised": True, "dtype": "float16",
     "n_reviews": len(work), "ids_hash": ids_hash, "mode": MODE}, indent=2))

n_chunks = -(-len(work) // TC.chunk_size)
todo = [c for c in range(n_chunks) if not (EMB_DIR / f"chunk_{c:03d}.npy").exists()]
print(f"{n_chunks} chunk(s); {n_chunks - len(todo)} already on Drive, {len(todo)} to encode")
if todo and RUN_FULL and DEVICE != "cuda":
    raise RuntimeError("Embedding every review needs a GPU: Runtime -> Change runtime type -> T4 GPU.")
if todo:
    encoder = load_encoder(MODEL_NAME)
    t_start = time.time()
    for i, c in enumerate(todo, start=1):
        texts = work["text"].iloc[c * TC.chunk_size:(c + 1) * TC.chunk_size].tolist()
        emb = encode(encoder, texts)
        tmp = EMB_DIR / f"chunk_{c:03d}.tmp"
        with open(tmp, "wb") as f:                     # write then rename, so a crash never leaves half a chunk
            np.save(f, emb)
        os.replace(tmp, EMB_DIR / f"chunk_{c:03d}.npy")
        per = (time.time() - t_start) / i
        print(f"  chunk {c + 1}/{n_chunks}: {len(texts):,} reviews | ETA {per * (len(todo) - i) / 60:.1f} min")
    del encoder
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

E = np.concatenate([np.load(EMB_DIR / f"chunk_{c:03d}.npy") for c in range(n_chunks)])
print(f"Embeddings: {E.shape[0]:,} × {E.shape[1]} ({E.nbytes / 1e9:.2f} GB, float16) in {EMB_DIR}")

13 chunk(s); 13 already on Drive, 0 to encode
Embeddings: 608,598 × 1024 (1.25 GB, float16) in /content/drive/MyDrive/bt4012_yelp/artifacts/embeddings/full/BAAI__bge-m3


### T1.3 Embedding checks
- **Shape and values:** one row per review, no NaN, unit length (within float16 rounding).
- **Exact copies:** reviews whose text is identical to another account's (`copy_other_accounts > 0`, from 1.3) should have similarity ≈ 1 with their copy. This confirms rows line up with review IDs.
- **Random pairs:** the typical similarity between unrelated reviews. Embedding models rarely centre this at 0, so T2's cutoffs are read relative to it.

In [8]:
assert E.shape[0] == len(work), "embedding rows do not match the working set"
norms = np.concatenate([np.linalg.norm(E[s:s + 100_000].astype(np.float32), axis=1)
                        for s in range(0, len(E), 100_000)])          # chunked: a float32 copy of all of E is 2.5 GB
assert np.isfinite(norms).all(), "embeddings contain NaN or inf"
print(f"Vector length: min {norms.min():.3f}, max {norms.max():.3f} (should be ~1)")


def pair_sims(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return (E[a].astype(np.float32) * E[b].astype(np.float32)).sum(axis=1)


copy_pos = (work.loc[work["copy_other_accounts"].gt(0), ["text_key"]].reset_index()
            .groupby("text_key")["index"].agg(["first", "last", "size"]))
copy_pos = copy_pos[copy_pos["size"] >= 2]
if len(copy_pos):
    s = pair_sims(copy_pos["first"].to_numpy(), copy_pos["last"].to_numpy())
    print(f"Exact copies: {len(s):,} pairs, similarity median {np.median(s):.3f}, min {s.min():.3f}")
    if np.median(s) < 0.98:
        print("WARNING: exact copies are not near-identical; rows may be misaligned with review IDs.")
else:
    print("No exact-copy pairs in this working set (expected on small debug samples).")

rng = np.random.default_rng(TC.seed)
ra, rb = rng.integers(0, len(work), 5_000), rng.integers(0, len(work), 5_000)
rs = pair_sims(ra[ra != rb], rb[ra != rb])
print(f"Random pairs: similarity mean {rs.mean():.3f}, 99th percentile {np.quantile(rs, 0.99):.3f}")

Vector length: min 1.000, max 1.001 (should be ~1)
Exact copies: 368 pairs, similarity median 1.000, min 0.992
Random pairs: similarity mean 0.569, 99th percentile 0.712


## T2. Similarity features
**Objective:** turn embeddings into features that catch near-duplicate and coordinated text across accounts, which the exact-copy hash in 1.3 misses (paraphrases, templates with a name swapped).

| Feature | Level | Meaning | Used by |
| --- | --- | --- | --- |
| `max_sim_other_account` | review | Similarity to the closest review by a **different** reviewer, anywhere | D2, `find_similar_reviews` |
| `nearest_other_review_id` | review | That closest review (the pointer the agent follows) | `find_similar_reviews`, LLM |
| `near_dup_accounts` | review | Distinct other reviewers with a review at similarity ≥ τ (among the 20 nearest; capped at 20) | D2 |
| `near_copy_accounts` | review | The same count at the stricter `tau_copy` = 0.97, where matches read as copying rather than generic wording | D2, LLM |
| `max_sim_same_restaurant` | review | Closest review by a different reviewer **at the same restaurant** (coordinated campaigns) | D2, D4 (optional) |
| `reviewer_self_similarity` | reviewer | Mean similarity among the reviewer's own reviews; blank for one-review accounts | D2 (the doc's self-similarity feature) |

Plus `neighbours.parquet`: each review's 10 nearest other-account reviews with similarities, which `find_similar_reviews` reads directly (no search index needed in the pipeline).

None of these use labels except the choice of τ in T2.3, which uses the training split only.

### T2.1 Nearest other-account neighbours
Exact search: the embeddings are multiplied block by block on the GPU (about a minute for all reviews on a T4), with each reviewer's own reviews masked out, keeping the top `knn_k` per review. Results are saved, so a later full run on a CPU runtime (with `EMBED_MODEL` pinned) reloads them and re-derives T2.2–T2.4 without the GPU.

In [9]:
KNN_SIM, KNN_IDX = EMB_DIR / "knn_sim.npy", EMB_DIR / "knn_idx.npy"
rcode = work["reviewer_id"].factorize()[0].astype(np.int64)
K = min(TC.knn_k, len(work) - 1)

dtype = torch.float16 if DEVICE == "cuda" else torch.float32     # half-precision matmul is slow on CPU
Et = torch.from_numpy(E).to(DEVICE, dtype=dtype)
rc = torch.from_numpy(rcode).to(DEVICE)

if KNN_SIM.exists() and KNN_IDX.exists() and np.load(KNN_IDX, mmap_mode="r").shape == (len(work), K):
    top_sim, top_idx = np.load(KNN_SIM), np.load(KNN_IDX)
    print("Loaded saved neighbours.")
else:
    if RUN_FULL and DEVICE != "cuda":
        raise RuntimeError("Searching every review needs a GPU: Runtime -> Change runtime type -> T4 GPU.")
    top_sim = np.empty((len(work), K), dtype=np.float32)
    top_idx = np.empty((len(work), K), dtype=np.int32)
    t0 = time.time()
    with torch.no_grad():
        for s in range(0, len(work), TC.knn_block):
            sims = Et[s:s + TC.knn_block] @ Et.T
            sims.masked_fill_(rc[s:s + TC.knn_block, None] == rc[None, :], -2.0)   # own reviews (incl. itself)
            v, i = sims.topk(K, dim=1)
            top_sim[s:s + len(v)], top_idx[s:s + len(v)] = v.float().cpu().numpy(), i.cpu().numpy()
    np.save(KNN_SIM, top_sim)
    np.save(KNN_IDX, top_idx)
    print(f"Searched {len(work):,} reviews in {time.time() - t0:.0f}s")

valid = top_sim > -1.5          # -2 marks "no other-account review left" (only in tiny samples)
print(f"Nearest other-account similarity: median {np.median(top_sim[:, 0]):.3f}, "
      f"99th percentile {np.quantile(top_sim[:, 0], 0.99):.3f}")

Loaded saved neighbours.
Nearest other-account similarity: median 0.818, 99th percentile 0.942


### T2.2 Same-restaurant similarity and reviewer self-similarity
- **Same restaurant:** for each restaurant, all its reviews are compared with each other (other reviewers only). Restaurants are small enough to do this exactly.
- **Self-similarity:** the mean pairwise similarity of a reviewer's own reviews, computed from the sum of their vectors without forming the pairs: for unit vectors, the sum of all pairwise similarities is ‖Σv‖² − Σ‖v‖².

In [10]:
# Same restaurant, other reviewers
rest = work["restaurant_id"].factorize()[0]
order = np.argsort(rest, kind="stable")
bounds = np.flatnonzero(np.diff(rest[order])) + 1
starts, ends = np.r_[0, bounds], np.r_[bounds, len(order)]
order_t = torch.from_numpy(order).to(DEVICE)
max_same = np.full(len(work), np.nan, dtype=np.float32)
t0 = time.time()
with torch.no_grad():
    for a, b in zip(starts, ends):
        if b - a < 2:
            continue
        idx = order_t[a:b]
        for s in range(0, b - a, TC.knn_block):                  # blocks keep very large restaurants in memory
            q = idx[s:s + TC.knn_block]
            sims = Et[q] @ Et[idx].T
            sims.masked_fill_(rc[q, None] == rc[idx][None, :], -2.0)
            m = sims.max(dim=1).values.float().cpu().numpy()
            max_same[order[a + s:a + s + len(q)]] = np.where(m > -1.5, m, np.nan)
print(f"Same-restaurant similarity done in {time.time() - t0:.0f}s; "
      f"{np.isnan(max_same).mean():.1%} of reviews have no other reviewer at their restaurant in this set")

# Reviewer self-similarity
n_rev = int(rcode.max()) + 1
S = torch.zeros((n_rev, E.shape[1]), dtype=torch.float32, device=DEVICE)
sq = torch.zeros(n_rev, dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    for s in range(0, len(work), 100_000):
        v = Et[s:s + 100_000].float()
        S.index_add_(0, rc[s:s + 100_000], v)
        sq.index_add_(0, rc[s:s + 100_000], (v * v).sum(dim=1))
counts = np.bincount(rcode, minlength=n_rev).astype(np.float64)
pair_sum = (S * S).sum(dim=1).cpu().numpy() - sq.cpu().numpy()
with np.errstate(invalid="ignore", divide="ignore"):
    self_sim_rev = np.where(counts >= 2, pair_sum / (counts * (counts - 1)), np.nan)
self_sim = self_sim_rev[rcode].astype(np.float32)
del S, sq
print(f"Self-similarity: {np.isfinite(self_sim_rev).sum():,} reviewers with 2+ reviews, "
      f"median {np.nanmedian(self_sim_rev):.3f}")

Same-restaurant similarity done in 2s; 0.1% of reviews have no other reviewer at their restaurant in this set
Self-similarity: 90,179 reviewers with 2+ reviews, median 0.603


### T2.3 Choose τ on the training split and save the features
τ defines a "near duplicate". It is chosen like D1's cutoffs: for each candidate, the training reviews whose closest other-account review is at least τ similar, and their fraud lift. The rule is the broadest τ with lift ≥ `min_lift` (1.5×), falling back to the highest-lift τ. Only training labels are used.

A second, fixed cutoff `tau_copy` = 0.97 gives `near_copy_accounts`. Matches between τ and 0.97 are mostly generic wording rather than copying (see T2 conclusions), so the two counts let D2 test which meaning carries the signal.

The features are then saved for the pipeline, and the 10 nearest neighbours per review for `find_similar_reviews`.

In [11]:
review_ids = work["review_id"].to_numpy()
max_sim = np.where(valid[:, 0], top_sim[:, 0], np.nan).astype(np.float32)
train = work["split"].eq("train").to_numpy()
base = work.loc[train, "is_fraud"].mean()

tau_rows = []
for tau in TC.tau_grid:
    hit = train & (max_sim >= tau)
    prec = work.loc[hit, "is_fraud"].mean() if hit.any() else np.nan
    tau_rows.append({"tau": tau, "coverage": hit.sum() / train.sum(), "precision": prec, "lift": prec / base})
tau_table = pd.DataFrame(tau_rows).set_index("tau")
ok = tau_table[tau_table["lift"] >= TC.min_lift]
TAU = float(ok.index.min()) if len(ok) else float(tau_table["lift"].idxmax())
display(tau_table.style.format({"coverage": "{:.2%}", "precision": "{:.1%}", "lift": "{:.2f}×"})
        .apply(lambda r: ["background-color: #fde3d6" if r.name == TAU else "" for _ in r], axis=1))
print(f"τ = {TAU} (broadest with training lift ≥ {TC.min_lift}×)" if len(ok) else
      f"τ = {TAU} (no τ reaches {TC.min_lift}×; highest lift used)")


def count_near_dup_accounts(tau: float) -> np.ndarray:
    '''Distinct other reviewers among the K nearest with similarity >= tau.'''
    codes = np.where(valid & (top_sim >= tau), rcode[top_idx], -1)
    codes.sort(axis=1)
    new = np.ones_like(codes, dtype=bool)
    new[:, 1:] = codes[:, 1:] != codes[:, :-1]
    return (new & (codes >= 0)).sum(axis=1).astype(np.int16)


features = pd.DataFrame({
    "review_id": review_ids,
    "max_sim_other_account": max_sim,
    "nearest_other_review_id": pd.Series(review_ids[top_idx[:, 0]], dtype=object).where(valid[:, 0], None),
    "near_dup_accounts": count_near_dup_accounts(TAU),
    "near_copy_accounts": count_near_dup_accounts(TC.tau_copy),
    "max_sim_same_restaurant": max_same,
    "reviewer_self_similarity": self_sim,
})
FEATURES_PATH = DATA_DIR / f"similarity_features{SUFFIX}.parquet"
save_table(features, FEATURES_PATH, {
    "made_by": "train.ipynb T2", "embedding_model": MODEL_NAME, "max_seq_len": TC.max_seq_len,
    "knn_k": K, "tau": TAU, "tau_rule": f"broadest with training lift >= {TC.min_lift}", "tau_copy": TC.tau_copy,
    "tau_table": tau_table.reset_index().to_dict("records"),
    "notes": "near_dup_accounts is capped at knn_k; other-account neighbours only"})

n_keep = min(TC.n_neighbours_saved, K)
keep = valid[:, :n_keep]
neighbours = pd.DataFrame({
    "review_id": np.repeat(review_ids, n_keep)[keep.ravel()],
    "rank": np.tile(np.arange(1, n_keep + 1), len(work))[keep.ravel()],
    "neighbour_review_id": review_ids[top_idx[:, :n_keep]].ravel()[keep.ravel()],
    "similarity": top_sim[:, :n_keep].ravel()[keep.ravel()].round(4),
})
NEIGHBOURS_PATH = ARTIFACTS_DIR / f"neighbours{SUFFIX}.parquet"
save_table(neighbours, NEIGHBOURS_PATH, {"made_by": "train.ipynb T2", "embedding_model": MODEL_NAME,
                                         "neighbours_per_review": n_keep, "other_accounts_only": True})
print(f"Saved {FEATURES_PATH.name} ({len(features):,} rows) and {NEIGHBOURS_PATH.name} ({len(neighbours):,} rows)")

,coverage,precision,lift
tau,,,
0.800000,67.96%,13.4%,1.01×
0.850000,20.20%,16.8%,1.27×
0.900000,2.76%,30.9%,2.33×
0.930000,1.26%,37.3%,2.81×
0.950000,0.86%,39.4%,2.98×
0.970000,0.60%,42.0%,3.18×
0.990000,0.42%,44.1%,3.33×


τ = 0.9 (broadest with training lift ≥ 1.5×)
Saved similarity_features.parquet (608,598 rows) and neighbours.parquet (6,085,980 rows)


In [12]:
# ---- T2.3b Minimum review length for the near-duplicate counts ---------------------------
SIM_LENGTH_EDGES = [0, 8, 15, 20, 30, 50]       # words; band lower edges
MIN_CLOSE = 100                                 # bands with fewer close matches are too thin to judge

wc = work["word_count"].to_numpy()
y = work["is_fraud"].to_numpy()
close = valid[:, 0] & (top_sim[:, 0] >= TAU)

# 1) Within each length band (training split): are reviews with a close match riskier than the band as a whole?
rows = []
for lo, hi in zip(SIM_LENGTH_EDGES, SIM_LENGTH_EDGES[1:] + [np.inf]):
    band = train & (wc >= lo) & (wc < hi)
    hit = band & close
    rows.append({"words": f"{lo}–{hi - 1:.0f}" if np.isfinite(hi) else f"{lo}+", "min_words": lo,
                 "reviews": band.sum(), "close matches": hit.sum(),
                 "band fraud rate": y[band].mean(), "fraud rate if close match": y[hit].mean() if hit.any() else np.nan})
length_table = pd.DataFrame(rows)
length_table["within-band lift"] = length_table["fraud rate if close match"] / length_table["band fraud rate"]

# 2) Rule: smallest length from which every (well-populated) band keeps within-band lift >= min_lift
judged = length_table[length_table["close matches"] >= MIN_CLOSE]
passes = judged["within-band lift"] >= TC.min_lift
ok_from = [lo for lo in judged["min_words"] if passes[judged["min_words"] >= lo].all()]
if judged.empty or not ok_from:
    SIM_MIN_WORDS = 0          # nothing to judge (small debug runs), or even the longest band fails: no filter
    if not judged.empty:
        print("WARNING: no length from which close matches keep the lift floor; counts left unfiltered. Check the table.")
else:
    SIM_MIN_WORDS = int(min(ok_from))
display(length_table.drop(columns="min_words").set_index("words").style.format(
    {"reviews": "{:,}", "close matches": "{:,}", "band fraud rate": "{:.1%}",
     "fraud rate if close match": "{:.1%}", "within-band lift": "{:.2f}×"}, na_rep="-"))
print(f"Minimum length for near-duplicate counts: {SIM_MIN_WORDS} words (both reviews)")

# 3) Recount with the length filter on both reviews; max_sim_other_account is unchanged
long_enough = wc >= SIM_MIN_WORDS


def count_near_dup_accounts(tau: float) -> np.ndarray:
    '''Distinct other reviewers among the K nearest with similarity >= tau, both reviews >= SIM_MIN_WORDS.'''
    keep = valid & (top_sim >= tau) & long_enough[top_idx] & long_enough[:, None]
    codes = np.where(keep, rcode[top_idx], -1)
    codes.sort(axis=1)
    new = np.ones_like(codes, dtype=bool)
    new[:, 1:] = codes[:, 1:] != codes[:, :-1]
    return (new & (codes >= 0)).sum(axis=1).astype(np.int16)


before = features.loc[train, ["near_dup_accounts", "near_copy_accounts"]].groupby(y[train]).mean()
features["near_dup_accounts"] = count_near_dup_accounts(TAU)
features["near_copy_accounts"] = count_near_dup_accounts(TC.tau_copy)
after = features.loc[train, ["near_dup_accounts", "near_copy_accounts"]].groupby(y[train]).mean()
cmp = pd.concat({"before": before, "after": after}, axis=1)
cmp.index = ["genuine", "fraud"]
cmp.loc["fraud ÷ genuine"] = cmp.loc["fraud"] / cmp.loc["genuine"]
print("\nTraining means of the counts, before and after the length filter:")
display(cmp.style.format("{:.3f}"))

old_meta = json.loads(FEATURES_PATH.with_suffix(".meta.json").read_text())
for k in ["rows", "columns", "mode", "created_at"]:
    old_meta.pop(k, None)
save_table(features, FEATURES_PATH, {**old_meta, "sim_min_words": SIM_MIN_WORDS,
                                     "sim_min_words_rule": f"smallest length with within-band lift >= {TC.min_lift} "
                                                           f"in every band with >= {MIN_CLOSE} close matches",
                                     "length_table": length_table.to_dict("records")})
print(f"Re-saved {FEATURES_PATH.name} with length-filtered counts")

,reviews,close matches,band fraud rate,fraud rate if close match,within-band lift
words,,,,,
0–7,"8,859","4,208",30.2%,34.1%,1.13×
8–14,"13,328","2,608",24.9%,29.3%,1.18×
15–19,"12,940","1,153",20.0%,28.3%,1.41×
20–29,"28,276","1,110",18.5%,28.9%,1.57×
30–49,"58,725",840,16.6%,31.1%,1.87×
50+,"304,203","1,861",10.8%,28.6%,2.64×


Minimum length for near-duplicate counts: 20 words (both reviews)

Training means of the counts, before and after the length filter:


Re-saved similarity_features.parquet with length-filtered counts


### T2.4 Checks and a first look at the signal
- **Agreement with exact copies:** reviews with an exact copy at another account should almost all have `max_sim_other_account` ≥ 0.99.
- **Generalisation:** fraud rate by band of `max_sim_other_account`, on training and validation side by side. Similar lifts on both mean τ and the bands were not tuned to noise.
- **Eyeball check:** a few pairs just above τ that are *not* exact copies, to see what the model calls a near duplicate.

In [13]:
f = work[["review_id", "split", "is_fraud", "copy_other_accounts", "text"]].merge(features, on="review_id")

exact = f["copy_other_accounts"].gt(0)
if exact.any():
    print(f"Exact copies with max_sim_other_account ≥ 0.99: {f.loc[exact, 'max_sim_other_account'].ge(0.99).mean():.1%} "
          f"of {exact.sum():,}" + ("" if RUN_FULL else " (lower in debug runs: some copies fall outside the sample)"))

edges = [-1.0, 0.7, 0.8, 0.9, TAU, 0.99, 1.01]
edges = sorted(set(edges))
f["band"] = pd.cut(f["max_sim_other_account"], edges, right=False)
rows = {}
for split in ["train", "val"]:
    s = f[f["split"] == split]
    g = s.groupby("band", observed=True)["is_fraud"].agg(reviews="size", fraud_rate="mean")
    g["lift"] = g["fraud_rate"] / s["is_fraud"].mean()
    rows[split] = g
display(pd.concat(rows, axis=1).style.format("{:.2f}×", subset=[("train", "lift"), ("val", "lift")])
        .format("{:.1%}", subset=[("train", "fraud_rate"), ("val", "fraud_rate")])
        .format("{:,.0f}", subset=[("train", "reviews"), ("val", "reviews")]))

print("\nSelf-similarity and same-restaurant similarity, fraud vs genuine (training split):")
display(f[f["split"] == "train"].groupby("is_fraud")[["reviewer_self_similarity", "max_sim_same_restaurant",
                                                      "near_dup_accounts", "near_copy_accounts"]].mean().round(3))

near = f[(f["max_sim_other_account"] >= TAU) & (f["max_sim_other_account"] < 0.99)]
text_by_id = work.set_index("review_id")["text"]
print(f"\nExamples just above τ = {TAU} (not exact copies):")
for _, r in near.sample(min(3, len(near)), random_state=TC.seed).iterrows():
    print(f"\n  similarity {r['max_sim_other_account']:.3f} | filtered={r['is_fraud']}")
    print(f"  A: {r['text'][:200]!r}")
    print(f"  B: {text_by_id[r['nearest_other_review_id']][:200]!r}")

Exact copies with max_sim_other_account ≥ 0.99: 100.0% of 886



Self-similarity and same-restaurant similarity, fraud vs genuine (training split):


,reviewer_self_similarity,max_sim_same_restaurant,near_dup_accounts,near_copy_accounts
is_fraud,,,,
0,0.598,0.791,0.012,0.002
1,0.607,0.783,0.034,0.015



Examples just above τ = 0.9 (not exact copies):

  similarity 0.937 | filtered=1
  A: 'Great food, very friendly and accommodating service, and reasonable prices! \xa0Lovely place!'
  B: 'Great place. Loved their food. I had fantastic service and the prices were very reasonable.'

  similarity 0.905 | filtered=0
  A: "LOVE this place. I'm vegan and they clearly mark their vegan options (and have quite a few of them). Not to mention the food is delicious and the service is great too! I love bringing new people here "
  B: "I love this place! Great service, super friendly staff, awesome vegetarian options and really delicious food. Have been back several times and it's always consistent. Highly recommend."

  similarity 0.903 | filtered=0
  A: "Drinks are pretty good, food is ok as well. \xa0It sounds better on the menu than it actually is but that's ok. \xa0The service was so-so."
  B: "Food's alright. Drinks are pretty good. Service is usually just alright."


### T2.5 Hand-off to the pipeline
- **`data/similarity_features.parquet`** joins onto the pipeline's review table by `review_id`, before D2's features are built. It adds D2's self-similarity feature and the cross-account similarity features; D4 can optionally use `max_sim_same_restaurant` or the neighbour pairs as extra ties between reviewers.
- **`artifacts/neighbours.parquet`** backs `find_similar_reviews(review_id)`: a lookup, so the pipeline needs no GPU and no search index. Exact copies still come from `text_key` (1.3).
- The embeddings themselves stay in `artifacts/embeddings/` and are not read by the pipeline.

**Next in this notebook:** T3 DistilBERT fine-tuning (D3) on the T4.

### T2 Conclusions
**The embeddings line up with the reviews.** All 886 exact copies have a closest other-account match at ≥ 0.99, so each embedding row matches its review ID. Searching all 608,598 reviews took 102 s on the T4.

**Similarity only means something near the top of the scale.** Embeddings of unrelated reviews already score high (median closest match 0.818), so most of the range carries no signal. Below 0.9 the fraud lift is 0.96×; above it, lift climbs steadily:

| Closest other-account similarity | Train lift | Val lift | Share of reviews |
| --- | --- | --- | --- |
| < 0.9 | 0.96× | 0.96× | ~97% |
| 0.90–0.99 | 2.15× | 2.10× | 2.3% |
| ≥ 0.99 (copies) | 3.33× | 3.16× | 0.4% |

τ = 0.9 is the broadest cutoff with ≥ 1.5× lift. Validation lifts match training, so the bands are not tuned to noise. A small group with *no* close match (< 0.7, 0.6% of reviews) is also riskier (1.3–1.5×), likely unusual or very short texts.

**Among short reviews, similarity measures generic wording, not copying.** The sampled pairs between 0.90 and 0.97 were short, generic praise. T2.3b confirmed it on the training split: in reviews under 15 words, a close match barely raises the fraud rate above the band's own rate (1.13–1.18×), because short reviews are already risky (25–30%) and nearly half the shortest ones have a close match anyway. The extra risk from a close match grows with length, to 1.57× at 20–29 words and 2.64× at 50+, where a close match means real copying. Reviews with a close match are about 30% fraud at any length; only the baseline changes.

The near-duplicate counts therefore only count pairs where both reviews have **20+ words** (the first band to clear 1.5×). This sharpens `near_copy_accounts` from 4.1× to 8.7× fraud vs genuine; `near_dup_accounts` drops slightly (3.3× → 2.9×) because the length signal it was carrying moves to D2's `word_count`, where it belongs.

**The features differ a lot in usefulness** (training means, fraud vs genuine):

| Feature | Genuine | Fraud | Ratio |
| --- | --- | --- | --- |
| `near_copy_accounts` (≥ 0.97, 20+ words) | 0.002 | 0.015 | 8.7× |
| `near_dup_accounts` (≥ 0.9, 20+ words) | 0.012 | 0.034 | 2.9× |
| `reviewer_self_similarity` | 0.598 | 0.607 | ≈ 1 |
| `max_sim_same_restaurant` | 0.791 | 0.783 | ≈ 1 |

The two counts are sharp but rare: they matter for the minority of reviews that are actually copied, not for the bulk. Self-similarity and same-restaurant similarity barely differ on average and need testing among 2+ reviewers and at the high end before they are kept.

**Implications**
- D2 takes `max_sim_other_account` (unfiltered, so D2 can learn its interaction with `word_count`) and the length-filtered `near_dup_accounts` and `near_copy_accounts`; self-similarity and same-restaurant similarity are kept only if they help on validation.
- `find_similar_reviews` should return the similarity with each match, and the agent's prompt should treat 0.90–0.97 as "similar wording" and only ≥ 0.97 between reviews of 20+ words as likely copying.
- Expect a modest gain in D2's PR-AUC from these features and more value as evidence for the agent, since copying is rare but easy to explain.

## T3. DistilBERT text detector (D3)
**Objective:** estimate each review's fraud probability from its words alone, by fine-tuning DistilBERT on Yelp's filter labels. It replaces the pipeline's TF-IDF stand-in for D3 only if it beats it on validation PR-AUC.

| Step | What it does | Runtime (full run, T4) |
| --- | --- | --- |
| T3.1 | Stratified ~100k training subsample, class weight, tokenisation | ~2 min |
| T3.2 | Fine-tune for 2 epochs with a checkpoint to Drive after each; validation PR-AUC per epoch | ~15–30 min per epoch (estimate) |
| T3.3 | Score every review with the best epoch; save raw logits for the pipeline | ~10–15 min (estimate) |

T3 does not use T1–T2: DistilBERT reads the raw text and learns its own representation. It only needs T0. Debug on CPU first (`RUN_FULL = False`: about 600 training and 300 validation reviews, one epoch), then run on a T4 with `RUN_FULL = True`.

### T3.1 Settings, training subsample and tokenisation
- **Subsample:** about `n_train` training reviews, stratified on the label so the 13% fraud rate is kept. Validation is used in full to choose the epoch; test is never touched.
- **Class imbalance:** the loss weights fraud by `pos_weight` = genuine ÷ fraud in the subsample (≈ 6.6), as in the design doc. Reviews are not resampled.
- **Tokenisation:** reviews are cut at `max_len` = 256 tokens; the cell reports how many are truncated. Batches are padded only to their own longest review.

In [14]:
# ---- T3.1 Settings, training subsample, tokenisation ----------------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import average_precision_score, roc_auc_score


@dataclass
class D3Config:
    base_model: str = "distilbert-base-uncased"
    n_train: int = 100_000          # stratified training subsample
    epochs: int = 2
    max_len: int = 256              # tokens
    batch_size: int = 32
    eval_batch_size: int = 256
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_frac: float = 0.06
    log_every: int = 500            # training steps between progress lines
    score_chunk: int = 50_000       # reviews per saved chunk when scoring (resume unit)
    # CPU debug caps
    debug_train_rows: int = 600
    debug_val_rows: int = 300
    debug_epochs: int = 1


DC = D3Config()
CPU_DEBUG = DEVICE == "cpu" and not RUN_FULL      # a full run on CPU can only reuse saved checkpoints and scores

EPOCHS = DC.debug_epochs if CPU_DEBUG else DC.epochs
D3_DIR = ARTIFACTS_DIR / "distilbert" / MODE
D3_DIR.mkdir(parents=True, exist_ok=True)

train_pool, val_pool = work[work["split"] == "train"], work[work["split"] == "val"]
n_train = min(DC.debug_train_rows if CPU_DEBUG else DC.n_train, len(train_pool))
tr = (train_pool.groupby("is_fraud", group_keys=False)
      .sample(frac=n_train / len(train_pool), random_state=TC.seed).reset_index(drop=True))
va = (val_pool.sample(min(DC.debug_val_rows, len(val_pool)), random_state=TC.seed) if CPU_DEBUG
      else val_pool).reset_index(drop=True)
pos_weight = (1 - tr["is_fraud"].mean()) / tr["is_fraud"].mean()

reviews_per_reviewer = d["reviewer_id"].value_counts()          # over all reviews, not just the working set
va_one = va["reviewer_id"].map(reviews_per_reviewer).eq(1).to_numpy()

tokenizer = AutoTokenizer.from_pretrained(DC.base_model)


def tokenize(texts) -> list[np.ndarray]:
    enc = tokenizer(list(texts), truncation=True, max_length=DC.max_len)["input_ids"]
    return [np.asarray(x, dtype=np.int32) for x in enc]


t0 = time.time()
tr_ids, va_ids = tokenize(tr["text"]), tokenize(va["text"])
lens = np.array([len(x) for x in tr_ids])
print(f"Training subsample: {len(tr):,} reviews ({tr['is_fraud'].mean():.1%} fraud), pos_weight {pos_weight:.2f}")
print(f"Validation: {len(va):,} reviews ({va['is_fraud'].mean():.1%} fraud); epochs: {EPOCHS}")
print(f"Tokenised in {time.time() - t0:.0f}s; tokens per review: median {np.median(lens):.0f}, "
      f"truncated at {DC.max_len}: {(lens >= DC.max_len).mean():.1%}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Training subsample: 100,000 reviews (13.2% fraud), pos_weight 6.55
Validation: 90,530 reviews (13.3% fraud); epochs: 2
Tokenised in 83s; tokens per review: median 111, truncated at 256: 15.9%


### T3.2 Fine-tune with a checkpoint every epoch
A plain PyTorch loop: AdamW with linear warm-up and decay, mixed precision on the GPU, gradient clipping, weighted binary cross-entropy on a single output logit.

After each epoch the cell scores validation and saves the model (`epoch_<k>/`) plus the full training state (`last.pt`) to Drive. If the session drops, re-run T0, T3.1 and this cell: it resumes from the last finished epoch. It refuses to resume if the settings or the subsample changed.

**Choosing the epoch:** the one with the highest validation PR-AUC. Training PR-AUC is not shown because the subsample was trained on.

In [15]:
# ---- T3.2 Fine-tune DistilBERT (resumable per epoch) ---------------------------------
AMP = DEVICE == "cuda"


def pad(seqs: list[np.ndarray]) -> tuple[torch.Tensor, torch.Tensor]:
    L = max(len(s) for s in seqs)
    ids = np.full((len(seqs), L), tokenizer.pad_token_id, dtype=np.int64)
    mask = np.zeros((len(seqs), L), dtype=np.int64)
    for i, s in enumerate(seqs):
        ids[i, :len(s)], mask[i, :len(s)] = s, 1
    return torch.from_numpy(ids).to(DEVICE), torch.from_numpy(mask).to(DEVICE)


@torch.no_grad()
def predict(model, ids: list[np.ndarray]) -> np.ndarray:
    '''Raw logits in the input order; batches are sorted by length so padding is minimal.'''
    model.eval()
    order = np.argsort([len(x) for x in ids], kind="stable")
    out = np.empty(len(ids), dtype=np.float32)
    for s in range(0, len(order), DC.eval_batch_size):
        idx = order[s:s + DC.eval_batch_size]
        input_ids, mask = pad([ids[i] for i in idx])
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
            out[idx] = model(input_ids=input_ids, attention_mask=mask).logits.squeeze(-1).float().cpu().numpy()
    return out


def val_metrics(logits: np.ndarray) -> dict:
    y = va["is_fraud"].to_numpy()
    sub = lambda m: average_precision_score(y[m], logits[m]) if m.any() and y[m].min() != y[m].max() else np.nan
    return {"val pr_auc": average_precision_score(y, logits), "val roc_auc": roc_auc_score(y, logits),
            "val pr_auc one-review": sub(va_one), "val pr_auc 2+ reviews": sub(~va_one)}


signature = {"base_model": DC.base_model, "max_len": DC.max_len, "batch_size": DC.batch_size, "lr": DC.lr,
             "epochs": EPOCHS, "train_ids": hashlib.sha1(tr["review_id"].astype(str).str.cat(sep="|").encode()).hexdigest()}
model = AutoModelForSequenceClassification.from_pretrained(DC.base_model, num_labels=1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=DC.lr, weight_decay=DC.weight_decay)
steps_per_epoch = -(-len(tr_ids) // DC.batch_size)
scheduler = get_linear_schedule_with_warmup(optimizer, int(DC.warmup_frac * steps_per_epoch * EPOCHS),
                                            steps_per_epoch * EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=AMP)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
y_tr = tr["is_fraud"].to_numpy(dtype=np.float32)

STATE = D3_DIR / "last.pt"
start_epoch, history = 0, []
if STATE.exists():
    ck = torch.load(STATE, map_location=DEVICE, weights_only=False)
    if ck["signature"] != signature:
        raise RuntimeError(f"{STATE} was saved with different settings or a different subsample; delete "
                           f"{D3_DIR} to start over.")
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, history = ck["epoch"], ck["history"]
    print(f"Resuming after epoch {start_epoch} of {EPOCHS}")

if start_epoch < EPOCHS and RUN_FULL and DEVICE != "cuda":
    raise RuntimeError(f"{EPOCHS - start_epoch} epoch(s) still to train; that needs a GPU: Runtime -> Change runtime type -> T4 GPU.")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    order = np.random.default_rng(TC.seed + epoch).permutation(len(tr_ids))
    t0, running = time.time(), []
    for step, s in enumerate(range(0, len(order), DC.batch_size), start=1):
        idx = order[s:s + DC.batch_size]
        input_ids, mask = pad([tr_ids[i] for i in idx])
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
            logits = model(input_ids=input_ids, attention_mask=mask).logits.squeeze(-1)
        loss = loss_fn(logits.float(), torch.from_numpy(y_tr[idx]).to(DEVICE))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running.append(loss.item())
        if step % DC.log_every == 0 or step == steps_per_epoch:
            per = (time.time() - t0) / step
            print(f"  epoch {epoch + 1} step {step:,}/{steps_per_epoch:,} | loss {np.mean(running[-DC.log_every:]):.4f} "
                  f"| ETA {per * (steps_per_epoch - step) / 60:.1f} min")
    row = {"epoch": epoch + 1, "train loss": float(np.mean(running)), "minutes": (time.time() - t0) / 60,
           **val_metrics(predict(model, va_ids))}
    history.append(row)
    print(f"Epoch {epoch + 1}: " + ", ".join(f"{k} {v:.3f}" for k, v in row.items() if k != "epoch"))

    model.save_pretrained(D3_DIR / f"epoch_{epoch + 1}")
    tokenizer.save_pretrained(D3_DIR / f"epoch_{epoch + 1}")
    tmp = D3_DIR / "last.tmp"
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch + 1, "history": history, "signature": signature}, tmp)
    os.replace(tmp, STATE)

d3_history = pd.DataFrame(history).set_index("epoch")
display(d3_history.style.format("{:.3f}").highlight_max(subset=["val pr_auc"], color="#fde3d6"))
BEST_EPOCH = int(d3_history["val pr_auc"].idxmax())
print(f"Best epoch by validation PR-AUC: {BEST_EPOCH}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Resuming after epoch 2 of 2


,train loss,minutes,val pr_auc,val roc_auc,val pr_auc one-review,val pr_auc 2+ reviews
epoch,,,,,,
1,1.070,10.559,0.318,0.750,0.457,0.177
2,0.963,10.590,0.325,0.755,0.467,0.176


Best epoch by validation PR-AUC: 2


### T3.3 Score every review and hand off to the pipeline
Loads the best epoch and scores every review in the working set, in chunks saved to Drive (a dropped session resumes at the next chunk). The raw logit is saved as `score` through `save_raw_scores`; the pipeline calibrates it on validation and picks the threshold, exactly as for D1.

Scores for the ~100k training reviews in the subsample are in-sample. That is fine for the stacker, which is fitted on validation, but D3 must never be judged on training reviews. The check below compares PR-AUC on the subsample, on training reviews outside it, and on validation; a large gap between the first and the others shows overfitting.

In [16]:
# ---- T3.3 Score every review with the best epoch -----------------------------------
best_model = AutoModelForSequenceClassification.from_pretrained(D3_DIR / f"epoch_{BEST_EPOCH}").to(DEVICE)
if AMP:
    best_model.half()
SCORE_DIR = D3_DIR / f"scores_epoch_{BEST_EPOCH}"
SCORE_DIR.mkdir(exist_ok=True)

n_chunks = -(-len(work) // DC.score_chunk)
missing = [c for c in range(n_chunks) if not (SCORE_DIR / f"chunk_{c:03d}.npy").exists()]
if missing and RUN_FULL and DEVICE != "cuda":
    raise RuntimeError(f"{len(missing)} chunk(s) still to score; that needs a GPU: Runtime -> Change runtime type -> T4 GPU.")
t0 = time.time()

for c in range(n_chunks):
    path = SCORE_DIR / f"chunk_{c:03d}.npy"
    if path.exists():
        continue
    texts = work["text"].iloc[c * DC.score_chunk:(c + 1) * DC.score_chunk]
    logits = predict(best_model, tokenize(texts))
    tmp = SCORE_DIR / f"chunk_{c:03d}.tmp"
    with open(tmp, "wb") as f:
        np.save(f, logits)
    os.replace(tmp, path)
    print(f"  chunk {c + 1}/{n_chunks} scored | {(time.time() - t0) / 60:.1f} min so far")
d3_scores = np.concatenate([np.load(SCORE_DIR / f"chunk_{c:03d}.npy") for c in range(n_chunks)])

d3_out = pd.DataFrame({"review_id": work["review_id"].to_numpy(), "score": d3_scores})
path = save_raw_scores("d3_distilbert", d3_out, {
    "made_by": "train.ipynb T3", "score_meaning": "DistilBERT logit (higher = more likely filtered); uncalibrated",
    "base_model": DC.base_model, "best_epoch": BEST_EPOCH, "n_train": len(tr), "pos_weight": pos_weight,
    "settings": asdict(DC), "history": d3_history.reset_index().to_dict("records"),
    "in_sample_review_ids": "the training subsample (see d3_train_subsample.parquet); do not evaluate on these"})
tr[["review_id"]].to_parquet(D3_DIR / "d3_train_subsample.parquet", index=False)
print(f"Saved {path}")

# Overfitting check: subsample (in-sample) vs other training reviews vs validation
s = work[["review_id", "split", "is_fraud"]].assign(score=d3_scores)
s["group"] = np.where(s["review_id"].isin(tr["review_id"]), "train subsample (in-sample)",
                      np.where(s["split"] == "train", "train, not in subsample", s["split"]))
rows = {}
for name, g in s[s["split"] != "test"].groupby("group"):
    rows[name] = {"reviews": len(g), "fraud rate": g["is_fraud"].mean(),
                  "pr_auc": average_precision_score(g["is_fraud"], g["score"]) if g["is_fraud"].nunique() == 2 else np.nan}
display(pd.DataFrame(rows).T.style.format({"reviews": "{:,.0f}", "fraud rate": "{:.1%}", "pr_auc": "{:.3f}"}))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Saved /content/drive/MyDrive/bt4012_yelp/artifacts/raw_scores/d3_distilbert.parquet


,reviews,fraud rate,pr_auc
train subsample (in-sample),"100,000",13.2%,0.443
"train, not in subsample","326,331",13.2%,0.329
val,"90,530",13.3%,0.325


### T3.4 Hand-off
- **`artifacts/raw_scores/d3_distilbert.parquet`** (+ `.meta.json`): one raw logit per review. In the pipeline, a small wrapper detector reads it; `run_detector()` calibrates it on validation, picks the threshold and reports it next to TF-IDF, D1 and d0.
- **`artifacts/distilbert/full/epoch_<k>/`**: the fine-tuned model and tokenizer, for SHAP (T5) or re-scoring.
- **`d3_train_subsample.parquet`**: the review IDs D3 was trained on; evaluation must exclude them (the test split never overlaps).



### T3 Conclusions
- **D3 ranks fraud about as well as D1, from the words alone.** Validation PR-AUC is 0.325 (2.4× the 0.133 base rate; ROC-AUC 0.755), level with D1 (0.318) and above d0 (0.272), without seeing who wrote the review.
- **Text is strongest for one-review accounts.** PR-AUC there is 0.467 against D1's 0.366 and D2's 0.433: where there is no history, the words are the main evidence. Among 2+ reviewers it is 0.176, no better than D1 and well behind D2 (0.350), so text and behaviour divide the work.
- **It generalises, with expected memorisation.** Training reviews outside the 100k subsample score 0.329 and validation 0.325; the subsample itself scores 0.443. D3 must never be evaluated on `d3_train_subsample.parquet`.
- **Two epochs were enough.** Epoch 2 improved validation PR-AUC only slightly (0.318 → 0.325) while training loss kept falling, so a third epoch would mostly add overfitting.
- **Limitation:** 15.9% of reviews exceed 256 tokens and are truncated. They are mostly long, genuine reviews, and the model still sees that they fill the limit, so little signal is likely lost.
- **Cost:** about 10.6 minutes per epoch and 15 minutes to score all 608,598 reviews on a T4, about 40 T4-minutes in total.

## T4. D2 1D CNN: the paper's model, replicated and tested as a D2 candidate
**Objective:** reproduce Zhang et al. (2023)'s behaviour-only CNN ("Conv1d" in their Table 4) and test whether it beats D2-GBM (validation PR-AUC 0.412 overall, 0.350 for 2+ reviewers). It replaces D2-GBM only if it beats it by about 0.01 or more; otherwise it stays a comparison in the report.

**What the paper specifies** (main text, Fig. 1–2, supplementary Appendices A, C, D)

| Choice | Paper | Used here |
| --- | --- | --- |
| Input | One vector per **reviewer**: 11 behaviour features (review count, tenure, review gap mean/std, rating entropy, absolute rating deviation mean/std, time since restaurant's first review mean/std, rating mean/std) | Run A: the same 11, one row per reviewer. Run B: D2's 26 features, one row per review |
| Feature order | Spearman correlation *between* features: highly correlated features are placed together as locally dependent. The exact procedure is not given; Appendix C shows their order beats ten random orders only slightly (AUC 0.792 vs 0.787–0.790) | Ours: features arranged so strongly correlated ones are neighbours (hierarchical clustering on 1 − abs Spearman, training data only) |
| Network | Conv1d: 64 filters, width 3, stride 1, ReLU → feature maps concatenated → Dense 128 → Dropout 0.5 → Dense 2 (Table A1, Fig. 1) | The same; the conv output is flattened into Dense 128 (the paper's 448-input dimension cannot be reconciled with 11 features), with ReLU after Dense 128 (not stated) |
| Data | Drop reviewers with < 3 reviews or mixed filtered/recommended labels, and restaurants with < 3 ratings (their Table 2 shows a minimum of 2 reviews, contradicting the text) | Reviewer filters as stated; the restaurant filter is approximated (features use all reviews) |
| Evaluation | 5-fold cross-validation; accuracy, precision, recall, F1, ROC-AUC. Conv1d: accuracy 80.5%, precision 0.736, recall 0.705, F1 0.720, AUC 0.792 | 5-fold CV over training reviewers using the existing folds; PR-AUC added |
| Learning rate | Benchmarks searched {3e-3, 4e-4, 5e-5} with Optuna; their own value not stated | Chosen from the same three values |
| Optimiser, batch size, epochs, class imbalance | Not stated. Their accuracy, precision and recall imply about 35% fake reviewers in their evaluation data, far above YelpZip's base rates, so their population or sampling differs from ours | Our choices: Adam; batch 256 (A) or 1,024 (B); up to 30 epochs, early stopping with patience 3; class-weighted loss (genuine ÷ fraud), as in every other model here |

**Run A: replication (the paper's method), at two thresholds.** The paper's text drops reviewers with fewer than 3 reviews, but its Table 2 shows a minimum of 2, so Run A is done twice: 3+ reviews and 2+ reviews (one row per reviewer with consistent labels; label = fake if all their reviews are filtered; for 2-review reviewers the single gap's spread is 0). The population tables compare each with the paper's Table 2 means and its implied fake share.

**Run B: D2 candidate (the paper's network, D2's protocol).** All 26 D2 features prepared as for D2-LR (log transforms, training medians, standardised), one row per review, every reviewer. For each learning rate: train on training folds 1–4, early-stop on fold 0 PR-AUC, score validation; keep the learning rate with the best validation PR-AUC (as D2-GBM's grid did). Refit on all training reviews for the early-stopped number of epochs, score every review, and save `artifacts/raw_scores/d2_cnn.parquet` (no evidence: a CNN cannot name the features behind a score). Compared with D2-LR and D2-GBM on validation by subgroup.



In [5]:
# ---- T4 D2 1D CNN: replication (Run A) and D2 candidate (Run B) ----------------------
import copy
import warnings
import torch.nn as nn
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import squareform
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, roc_auc_score

torch.set_num_threads(os.cpu_count() or 2)
DET_DIR = PROJECT_ROOT / "outputs" / "detectors"
LRS = [3e-3, 4e-4, 5e-5]                      # the paper's learning-rate search space
MAX_EPOCHS, PATIENCE = (30, 3) if RUN_FULL else (3, 1)
PAPER_11 = ["reviewer_n_reviews", "reviewer_active_days", "gap_mean", "gap_std", "rating_entropy", "abs_dev_mean",
            "abs_dev_std", "time_mean", "time_std", "rating_mean", "rating_std"]
PAPER_REPORTED = {"accuracy": 0.8051, "precision": 0.7356, "recall": 0.7047, "f1": 0.7198, "roc_auc": 0.7924}

# ---- 1. Load D2's features and the pipeline's preprocessing choices ------------------------
d2f = pd.read_parquet(DATA_DIR / "d2_features.parquet")
D2_COLS_T = json.loads((DATA_DIR / "d2_features.meta.json").read_text())["signature"]["model_columns"]   # D2's model inputs
LOG_T = json.loads((DET_DIR / "d2_lr.meta.json").read_text())["params"]["log_features"]
X = work[["review_id", "reviewer_id", "split", "fold", "is_fraud"]].merge(d2f, on="review_id", validate="one_to_one")
if len(X) != len(work):
    raise RuntimeError("d2_features.parquet does not cover the working set; re-run pipeline.ipynb 4.1.")
print(f"{len(X):,} reviews × {len(D2_COLS_T)} D2 features loaded")


def correlation_order(frame: pd.DataFrame, cols: list[str]) -> list[str]:
    '''Neighbouring positions = strongly correlated features (1 - |Spearman|, average linkage, optimal leaf order).'''
    sample = frame[cols].sample(min(len(frame), 100_000), random_state=TC.seed)
    rho = sample.corr(method="spearman").abs().fillna(0).to_numpy()
    dist = squareform(np.clip(1 - rho, 0, None), checks=False)
    return [cols[i] for i in leaves_list(optimal_leaf_ordering(linkage(dist, "average"), dist))]


def prepare(train: pd.DataFrame, others: list[pd.DataFrame], cols: list[str]) -> list[np.ndarray]:
    '''Log-transform skewed columns, fill blanks with training medians, standardise with training statistics.'''
    def raw(f):
        a = f[cols].to_numpy(dtype="float64")
        idx = [cols.index(c) for c in LOG_T if c in cols]
        a[:, idx] = np.log1p(a[:, idx])
        return a
    tr = raw(train)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                  # all-blank columns (e.g. in small debug samples)
        med = np.nanmedian(tr, axis=0)
    fill = lambda a: np.where(np.isnan(a), med, a)
    med = np.where(np.isnan(med), 0.0, med)             # a column blank throughout training
    tr = fill(tr)
    mu, sd = tr.mean(axis=0), tr.std(axis=0)
    sd[sd == 0] = 1.0
    return [((fill(raw(f)) - mu) / sd).astype(np.float32) for f in [train] + others]


# ---- 2. The paper's network and a training loop ---------------------------------------------
class PaperCNN(nn.Module):
    '''Conv1d (64 filters, width 3, stride 1) + ReLU -> flatten -> Dense 128 + ReLU -> Dropout 0.5 -> Dense 2.'''
    def __init__(self, n_features: int):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, kernel_size=3, stride=1), nn.ReLU(), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(64 * (n_features - 2), 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 2))

    def forward(self, x):
        return self.head(self.conv(x.unsqueeze(1)))


@torch.no_grad()
def cnn_score(model, Xa: np.ndarray) -> np.ndarray:
    '''Log-odds of "fake" (logit difference of the two outputs).'''
    model.eval()
    if len(Xa) == 0:
        return np.empty(0, dtype=np.float32)
    out = [model(torch.from_numpy(Xa[s:s + 65_536]).to(DEVICE)) for s in range(0, len(Xa), 65_536)]
    z = torch.cat(out).float().cpu().numpy()
    return z[:, 1] - z[:, 0]


def train_cnn(Xtr, ytr, lr, batch, Xstop=None, ystop=None, metric=average_precision_score,
              epochs=None, label="", verbose=False):
    '''Adam + class-weighted cross-entropy. With a stop set: early stopping on `metric` (patience PATIENCE),
    keeping the best epoch. Without: train exactly `epochs` epochs. Returns (model, best_epoch, best_metric).'''
    torch.manual_seed(TC.seed)
    model = PaperCNN(Xtr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    pos = float(np.clip(ytr.mean(), 1e-6, 1 - 1e-6))
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, (1 - pos) / pos], dtype=torch.float32, device=DEVICE))
    Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr.astype(np.int64))
    rng = np.random.default_rng(TC.seed)
    best, best_epoch, best_state, wait, t0 = -np.inf, 0, None, 0, time.time()
    n_epochs = epochs or MAX_EPOCHS
    for epoch in range(1, n_epochs + 1):
        model.train()
        order, losses = rng.permutation(len(Xtr)), []
        for s in range(0, len(order), batch):
            idx = torch.from_numpy(order[s:s + batch])
            loss = loss_fn(model(Xt[idx].to(DEVICE)), yt[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        if Xstop is None:
            best_epoch, best_state = epoch, None
            if verbose:
                print(f"    {label} epoch {epoch}/{n_epochs} | loss {np.mean(losses):.4f} | {time.time() - t0:.0f}s")
            continue
        m = metric(ystop, cnn_score(model, Xstop)) if len(np.unique(ystop)) == 2 else np.nan
        if verbose:
            per = (time.time() - t0) / epoch
            name = {"average_precision_score": "PR-AUC", "roc_auc_score": "ROC-AUC"}.get(metric.__name__, metric.__name__)
            print(f"    {label} epoch {epoch} | loss {np.mean(losses):.4f} | stop-set {name} "
                  f"{m:.4f} | {time.time() - t0:.0f}s (≤ {per * (n_epochs - epoch) / 60:.1f} min left)")
        if m > best:
            best, best_epoch, best_state, wait = m, epoch, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch, best


def class_metrics(y, s) -> dict:
    '''The paper's metrics at the model's own decision (fake if its output is the larger), plus PR-AUC.'''
    if len(y) == 0:
        return {k: np.nan for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]}
    both = len(np.unique(y)) == 2
    p, r, f, _ = precision_recall_fscore_support(y, s > 0, average="binary", zero_division=0)
    return {"accuracy": np.mean((s > 0) == y), "precision": p, "recall": r, "f1": f,
            "roc_auc": roc_auc_score(y, s) if both else np.nan, "pr_auc": average_precision_score(y, s) if both else np.nan}


# ---- 3. Run A: the paper's setup (reviewer level, 5-fold CV), at 3+ and at 2+ reviews ----------
# The paper's text says reviewers with fewer than 3 reviews were removed; its Table 2 shows a minimum of 2.
# Both populations are run; whichever matches the paper's Table 2 and implied fake share (~35%) better is the
# closer replication. Labels as in the paper: mixed-label reviewers dropped; fake = all reviews filtered.
PAPER_TABLE2_MEANS = {"reviewer_n_reviews": 3.99, "reviewer_active_days": 274.98, "gap_mean": 124.32, "gap_std": 46.56,
                      "rating_entropy": 0.86, "abs_dev_mean": 0.86, "abs_dev_std": 0.41, "time_mean": 1492.71,
                      "time_std": 549.70, "rating_mean": 3.94, "rating_std": 0.65}
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
rev_all = X.groupby("reviewer_id").agg(split=("split", "first"), fold=("fold", "first"), n=("review_id", "size"),
                                       fake_share=("is_fraud", "mean"), **{c: (c, "first") for c in PAPER_11})
# With exactly 2 reviews there is one gap, so its spread is 0 (the paper's Table 2 minimum), not blank
rev_all.loc[rev_all["n"] == 2, "gap_std"] = 0.0


def run_a(min_reviews: int) -> dict:
    rev = rev_all[(rev_all["n"] >= min_reviews) & rev_all["fake_share"].isin([0, 1])]
    rev = rev.assign(fake=rev["fake_share"].astype(int))
    rev_tr, rev_va = rev[rev["split"] == "train"], rev[rev["split"] == "val"]
    order = correlation_order(rev_tr, PAPER_11)
    print(f"\nRun A, {min_reviews}+ reviews: {len(rev_tr):,} training and {len(rev_va):,} validation reviewers; "
          f"fake share {rev_tr['fake'].mean():.1%} (the paper's figures imply about 35%)")
    print("  Feature order (correlated neighbours):", ", ".join(order))

    cv_rows, t0, n_runs = [], time.time(), len(LRS) * 5
    for lr in LRS:
        for k in range(5):
            test, stop = rev_tr[rev_tr["fold"] == k], rev_tr[rev_tr["fold"] == (k + 1) % 5]
            fit = rev_tr[~rev_tr["fold"].isin([k, (k + 1) % 5])]
            if len(fit) == 0 or len(test) == 0:          # only in tiny debug samples
                continue
            Xf, Xs, Xk = prepare(fit, [stop, test], order)
            model, ep, _ = train_cnn(Xf, fit["fake"].to_numpy(), lr, 256, Xs, stop["fake"].to_numpy(), metric=roc_auc_score)
            cv_rows.append({"lr": lr, "fold": k, "epochs": ep,
                            **class_metrics(test["fake"].to_numpy(), cnn_score(model, Xk))})
            done = len(cv_rows)
            print(f"  lr {lr:g} fold {k}: ROC-AUC {cv_rows[-1]['roc_auc']:.3f} | {done}/{n_runs} runs | "
                  f"{(time.time() - t0) / 60:.1f} min, ~{(time.time() - t0) / done * (n_runs - done) / 60:.1f} min left")
    cv = pd.DataFrame(cv_rows)
    lr_best = float(cv.groupby("lr")["roc_auc"].mean().idxmax())
    best_cv = cv[cv["lr"] == lr_best]

    # Validation check: train on folds 1-4, early-stop on fold 0, score validation reviewers of the same population
    fit, stop = rev_tr[rev_tr["fold"] != 0], rev_tr[rev_tr["fold"] == 0]
    Xf, Xs, Xv = prepare(fit, [stop, rev_va], order)
    model, _, _ = train_cnn(Xf, fit["fake"].to_numpy(), lr_best, 256, Xs, stop["fake"].to_numpy(), metric=roc_auc_score)
    return {"min_reviews": min_reviews, "learning_rate": lr_best, "order": order,
            "train reviewers": len(rev_tr), "train fake share": rev_tr["fake"].mean(),
            "val reviewers": len(rev_va), "val fake share": rev_va["fake"].mean(),
            "feature means": rev_tr[PAPER_11].mean(),
            "cv mean": best_cv[METRICS].mean(), "cv std": best_cv[METRICS].std(),
            "validation": pd.Series(class_metrics(rev_va["fake"].to_numpy(), cnn_score(model, Xv)))}


RUN_A = {m: run_a(m) for m in (3, 2)}

print("\nPopulation: who is kept, and how many are fake")
display(pd.DataFrame({f"{m}+ reviews": {k: r[k] for k in ["train reviewers", "train fake share", "val reviewers",
                                                           "val fake share", "learning_rate"]}
                      for m, r in RUN_A.items()}).T.assign(**{"paper (implied)": [0.355] * len(RUN_A)})
        .style.format({"train reviewers": "{:,.0f}", "val reviewers": "{:,.0f}", "train fake share": "{:.1%}",
                       "val fake share": "{:.1%}", "learning_rate": "{:g}", "paper (implied)": "{:.1%}"}))

print("Feature means (training reviewers) against the paper's Table 2")
display(pd.DataFrame({"paper Table 2": pd.Series(PAPER_TABLE2_MEANS),
                      **{f"ours, {m}+ reviews": r["feature means"] for m, r in RUN_A.items()}})
        .style.format("{:,.2f}"))

summary_a = pd.DataFrame({"paper (Conv1d, YelpZIP)": pd.Series(PAPER_REPORTED),
                          **{f"{m}+: CV mean": r["cv mean"] for m, r in RUN_A.items()},
                          **{f"{m}+: CV std": r["cv std"] for m, r in RUN_A.items()},
                          **{f"{m}+: validation": r["validation"] for m, r in RUN_A.items()}}).reindex(METRICS)
print("Metrics against the paper")
display(summary_a.style.format("{:.3f}", na_rep="-"))

# ---- 4. Run B: D2 candidate (review level, all reviewers, D2's 26 features) -----------------
tr_b, va_b = X[X["split"] == "train"], X[X["split"] == "val"]
ORDER_B = correlation_order(tr_b, D2_COLS_T)
fit_b, stop_b = tr_b[tr_b["fold"] != 0], tr_b[tr_b["fold"] == 0]
Xf, Xs, Xv = prepare(fit_b, [stop_b, va_b], ORDER_B)
print(f"\nRun B: {len(fit_b):,} training (folds 1-4), {len(stop_b):,} early-stopping (fold 0), {len(va_b):,} validation reviews")
print("Feature order (correlated neighbours):", ", ".join(ORDER_B))
lr_rows = []
for lr in LRS:
    print(f"  learning rate {lr:g}:")
    model, ep, stop_ap = train_cnn(Xf, fit_b["is_fraud"].to_numpy(), lr, 1024, Xs, stop_b["is_fraud"].to_numpy(),
                                   label=f"lr {lr:g}", verbose=True)
    lr_rows.append({"lr": lr, "epochs": ep, "fold-0 PR-AUC": stop_ap,
                    "val PR-AUC": average_precision_score(va_b["is_fraud"], cnn_score(model, Xv))})
lr_b = pd.DataFrame(lr_rows).set_index("lr")
LR_B, EPOCHS_B = float(lr_b["val PR-AUC"].idxmax()), int(lr_b.loc[lr_b["val PR-AUC"].idxmax(), "epochs"])
display(lr_b.style.format({"fold-0 PR-AUC": "{:.3f}", "val PR-AUC": "{:.3f}"}).highlight_max(subset=["val PR-AUC"], color="#fde3d6"))

print(f"\nRefitting on all {len(tr_b):,} training reviews: learning rate {LR_B:g}, {EPOCHS_B} epochs")
Xtr_all, Xwork = prepare(tr_b, [X], ORDER_B)
model_b, _, _ = train_cnn(Xtr_all, tr_b["is_fraud"].to_numpy(), LR_B, 1024, epochs=EPOCHS_B, label="refit", verbose=True)
CNN_DIR = ARTIFACTS_DIR / "d2_cnn"
CNN_DIR.mkdir(exist_ok=True)
torch.save(model_b.state_dict(), CNN_DIR / f"model{SUFFIX}.pt")
scores_b = cnn_score(model_b, Xwork)
path = save_raw_scores("d2_cnn", pd.DataFrame({"review_id": X["review_id"].to_numpy(), "score": scores_b}), {
    "made_by": "train.ipynb T4", "score_meaning": "1D CNN log-odds of fake (class-weighted; uncalibrated)",
    "architecture": "Conv1d(64, k=3, s=1)+ReLU -> Dense 128+ReLU -> Dropout 0.5 -> Dense 2 (Zhang et al. 2023, Table A1)",
    "feature_order": ORDER_B, "log_features": LOG_T, "learning_rate": LR_B, "epochs": EPOCHS_B,
    "lr_search": lr_b.reset_index().to_dict("records"),
    "run_a": {f"{m}+ reviews": {"train fake share": r["train fake share"], "learning_rate": r["learning_rate"],
                                "cv_mean": r["cv mean"].to_dict(), "validation": r["validation"].to_dict()}
              for m, r in RUN_A.items()}})
print(f"Saved {path}")

# ---- 5. Compare the three D2 candidates on validation -------------------------------------
n_rev = X["reviewer_n_reviews"].to_numpy()
cmp_frame = X[["review_id", "split", "is_fraud"]].assign(d2_cnn=scores_b, n=n_rev)
for name in ["d2_lr", "d2_gbm"]:
    cmp_frame = cmp_frame.merge(pd.read_parquet(DET_DIR / f"{name}.parquet", columns=["review_id", "score"])
                                .rename(columns={"score": name}), on="review_id", how="left")
v = cmp_frame[cmp_frame["split"] == "val"]
groups = {"all": v["n"] >= 1, "one-review": v["n"] == 1, "2+ reviews": v["n"] >= 2, "3+ reviews": v["n"] >= 3}
cmp = pd.DataFrame({g: {"base rate": v.loc[m, "is_fraud"].mean(),
                        **{name: average_precision_score(v.loc[m, "is_fraud"], v.loc[m, name])
                           for name in ["d2_lr", "d2_gbm", "d2_cnn"]}} for g, m in groups.items()}).T
print("\nValidation PR-AUC, the three D2 candidates:")
display(cmp.style.format("{:.3f}").highlight_max(subset=["d2_lr", "d2_gbm", "d2_cnn"], axis=1, color="#fde3d6"))

608,598 reviews × 22 D2 features loaded

Run A, 3+ reviews: 34,067 training and 7,322 validation reviewers; fake share 7.0% (the paper's figures imply about 35%)
  Feature order (correlated neighbours): abs_dev_mean, abs_dev_std, rating_std, rating_entropy, rating_mean, reviewer_n_reviews, reviewer_active_days, gap_std, gap_mean, time_mean, time_std
  lr 0.003 fold 0: ROC-AUC 0.805 | 1/15 runs | 0.1 min, ~2.1 min left
  lr 0.003 fold 1: ROC-AUC 0.813 | 2/15 runs | 0.2 min, ~1.2 min left
  lr 0.003 fold 2: ROC-AUC 0.811 | 3/15 runs | 0.2 min, ~0.9 min left
  lr 0.003 fold 3: ROC-AUC 0.811 | 4/15 runs | 0.2 min, ~0.7 min left
  lr 0.003 fold 4: ROC-AUC 0.796 | 5/15 runs | 0.3 min, ~0.6 min left
  lr 0.0004 fold 0: ROC-AUC 0.810 | 6/15 runs | 0.4 min, ~0.5 min left
  lr 0.0004 fold 1: ROC-AUC 0.813 | 7/15 runs | 0.4 min, ~0.4 min left
  lr 0.0004 fold 2: ROC-AUC 0.807 | 8/15 runs | 0.4 min, ~0.4 min left
  lr 0.0004 fold 3: ROC-AUC 0.812 | 9/15 runs | 0.5 min, ~0.3 min left
  lr 0.0004 fo

,train reviewers,train fake share,val reviewers,val fake share,learning_rate,paper (implied)
3+ reviews,"34,067",7.0%,"7,322",7.0%,0.0004,35.5%
2+ reviews,"61,630",10.7%,"13,218",10.8%,0.0004,35.5%


Feature means (training reviewers) against the paper's Table 2


,paper Table 2,"ours, 3+ reviews","ours, 2+ reviews"
reviewer_n_reviews,3.99,7.19,4.87
reviewer_active_days,274.98,569.40,403.43
gap_mean,124.32,134.37,162.96
gap_std,46.56,154.48,85.39
rating_entropy,0.86,1.18,0.93
abs_dev_mean,0.86,0.78,0.81
abs_dev_std,0.41,0.54,0.51
time_mean,"1,492.71","1,466.97","1,479.87"
time_std,549.70,780.37,727.93
rating_mean,3.94,3.95,3.97


Metrics against the paper


,"paper (Conv1d, YelpZIP)",3+: CV mean,2+: CV mean,3+: CV std,2+: CV std,3+: validation,2+: validation
accuracy,0.805,0.738,0.677,0.019,0.004,0.743,0.673
precision,0.736,0.173,0.213,0.009,0.002,0.176,0.210
recall,0.705,0.732,0.748,0.018,0.022,0.722,0.735
f1,0.720,0.280,0.332,0.011,0.004,0.283,0.327
roc_auc,0.792,0.808,0.769,0.006,0.007,0.812,0.763
pr_auc,-,0.246,0.267,0.019,0.008,0.261,0.276



Run B: 340,971 training (folds 1-4), 85,360 early-stopping (fold 0), 90,530 validation reviews
Feature order (correlated neighbours): copy_share, near_copy_share, near_dup_share, same_restaurant_sim_mean, reviewer_self_similarity, rating_entropy, rating_std, abs_dev_std, abs_dev_mean, extreme_share, rating_mean, length_mean, insufficient_history, rate_undefined, reviewer_n_reviews, reviewer_active_days, max_reviews_per_day, gap_mean, reviews_per_30d, gap_std, time_mean, time_std
  learning rate 0.003:
    lr 0.003 epoch 1 | loss 0.4944 | stop-set PR-AUC 0.3938 | 1s (≤ 0.5 min left)
    lr 0.003 epoch 2 | loss 0.4843 | stop-set PR-AUC 0.3908 | 3s (≤ 0.8 min left)
    lr 0.003 epoch 3 | loss 0.4813 | stop-set PR-AUC 0.3877 | 5s (≤ 0.7 min left)
    lr 0.003 epoch 4 | loss 0.4789 | stop-set PR-AUC 0.3898 | 6s (≤ 0.6 min left)
  learning rate 0.0004:
    lr 0.0004 epoch 1 | loss 0.4928 | stop-set PR-AUC 0.3941 | 1s (≤ 0.4 min left)
    lr 0.0004 epoch 2 | loss 0.4812 | stop-set PR-AUC 0.3

,epochs,fold-0 PR-AUC,val PR-AUC
lr,,,
0.003000,1,0.394,0.395
0.000400,5,0.397,0.407
0.000050,17,0.404,0.407



Refitting on all 426,331 training reviews: learning rate 5e-05, 17 epochs
    refit epoch 1/17 | loss 0.5139 | 2s
    refit epoch 2/17 | loss 0.4880 | 3s
    refit epoch 3/17 | loss 0.4844 | 4s
    refit epoch 4/17 | loss 0.4824 | 5s
    refit epoch 5/17 | loss 0.4805 | 5s
    refit epoch 6/17 | loss 0.4792 | 6s
    refit epoch 7/17 | loss 0.4789 | 7s
    refit epoch 8/17 | loss 0.4781 | 8s
    refit epoch 9/17 | loss 0.4767 | 9s
    refit epoch 10/17 | loss 0.4765 | 10s
    refit epoch 11/17 | loss 0.4757 | 11s
    refit epoch 12/17 | loss 0.4750 | 12s
    refit epoch 13/17 | loss 0.4744 | 14s
    refit epoch 14/17 | loss 0.4743 | 16s
    refit epoch 15/17 | loss 0.4738 | 17s
    refit epoch 16/17 | loss 0.4734 | 17s
    refit epoch 17/17 | loss 0.4729 | 18s
Saved /content/drive/MyDrive/bt4012_yelp/artifacts/raw_scores/d2_cnn.parquet

Validation PR-AUC, the three D2 candidates:


,base rate,d2_lr,d2_gbm,d2_cnn
all,0.133,0.363,0.412,0.411
one-review,0.298,0.382,0.433,0.431
2+ reviews,0.069,0.303,0.351,0.354
3+ reviews,0.048,0.299,0.359,0.369


### T4 conclusions
- **Replication (Run A) works.** On the paper's population (reviewers with 3+ reviews and consistent labels), the CNN reaches a 5-fold CV ROC-AUC of 0.808, against the paper's 0.792. Including 2-review accounts lowers it to 0.769. Our fake share is 7.0% (3+) and 10.7% (2+), against about 35% implied by the paper's metrics, so its accuracy, precision and recall cannot be compared directly with ours; ROC-AUC can.
- **As a D2 candidate (Run B), the CNN ties with D2-GBM.** Learning rate 5e-5, 17 epochs. Section 4.5 has the authoritative comparison, since it calibrates both models in the same way (the table printed here used D2-GBM's earlier 26-feature scores). Validation PR-AUC: 0.411 vs 0.415 overall, 0.431 vs 0.436 for one-review accounts, 0.354 vs 0.352 for 2+ and 0.369 vs 0.361 for 3+. All four differences are under the 0.01 bar, and the two models rank reviews almost identically (rank correlation 0.98).
- **Decision: D2 stays `d2_gbm`.** It is level or slightly ahead overall, and SHAP gives it per-review evidence, which the CNN cannot provide. The CNN stays in the report as the paper comparison; its saved scores (`d2_cnn`) are an optional stacker input but are expected to add little.
- **Takeaway:** the paper's model choice matters less than its inputs. Given the same 22 features, a CNN and gradient-boosted trees end up in the same place.